In [1]:
import os, sys
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ['OMP_NUM_THREADS'] = '1'
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if device.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

Device : cuda
GPU    : NVIDIA GeForce RTX 5060 Laptop GPU


In [2]:
import os

def is_colab():
    try: import google.colab; return True
    except ImportError: return False

if is_colab():
    from google.colab import drive
    drive.mount('/content/drive')
    COLAB_BASE = '/content/drive/MyDrive/Rakuten'
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'sentence-transformers', 'lightgbm'], check=False)
else:
    COLAB_BASE = None
print('Colab OK' if is_colab() else 'Local')

Local


In [3]:
LOCAL_BASE = r'C:/Users/veyss/Nouveau dossier/nov25_alt1_mle_rakuten'
BASE = COLAB_BASE if ('COLAB_BASE' in dir() and COLAB_BASE) else LOCAL_BASE
print(f'BASE : {BASE}')

BASE : C:/Users/veyss/Nouveau dossier/nov25_alt1_mle_rakuten


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle, time
import scipy.sparse as sp
from sklearn.metrics import f1_score, accuracy_score
from sklearn.preprocessing import normalize
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neural_network import MLPClassifier
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
from transformers import AutoModel
from tqdm import tqdm
from PIL import Image

c:\Users\veyss\anaconda3\envs\rakuten_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
SPLIT_DIR     = os.path.join(BASE, 'data', 'split_no_crop')
PROCESSED_DIR = os.path.join(BASE, 'data', 'processed')
MODELS_DIR    = os.path.join(BASE, 'models')
CACHE_DIR     = PROCESSED_DIR
OOF_DIR       = os.path.join(BASE, 'data', 'oof')
os.makedirs(OOF_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

In [6]:
train_split = pd.read_csv(os.path.join(SPLIT_DIR, 'train.csv'))
val_split   = pd.read_csv(os.path.join(SPLIT_DIR, 'val.csv'))
test_split  = pd.read_csv(os.path.join(SPLIT_DIR, 'test.csv'))

y_train = train_split['label'].values.astype(int)
y_val   = val_split['label'].values.astype(int)
y_test  = test_split['label'].values.astype(int)

with open(os.path.join(MODELS_DIR, 'label_encoder.pkl'), 'rb') as f:
    le = pickle.load(f)
class_names = [str(c) for c in le.inverse_transform(range(27))]

texts_train = train_split['text_combined_raw'].fillna('').tolist()
texts_val   = val_split['text_combined_raw'].fillna('').tolist()
texts_test  = test_split['text_combined_raw'].fillna('').tolist()

print(f'Splits : train={len(train_split):,} | val={len(val_split):,} | test={len(test_split):,}')

Splits : train=59,185 | val=12,993 | test=12,738


## Pourquoi OOF ?

Dans le stacking classique, les probas de niveau 1 calculees **sur le train** sont des predictions **in-sample** :
le modele a vu ces exemples pendant l'entrainement → les probas sont trop confiantes → le meta-classifieur
apprend un signal artificiel (le modele predit parfaitement ses propres exemples d'entrainement).

**OOF (Out-Of-Fold)** : on divise le train en K folds. Pour chaque fold k, le modele niveau-1
est entraine sur les K-1 autres folds et predit sur le fold k. On obtient ainsi des probas
pour TOUS les exemples du train sans jamais les avoir vus → pas de leakage.

Nota : pour le meta-classifieur, meme une tete lineaire sur backbone gele cause un leakage
(faible, mais reel). OOF le supprime completement.

## 1. Chargement des features pre-calculees

In [7]:
# ── Embeddings SentenceTransformer e5-large ───────────────────────────
ST_MODEL = 'intfloat/multilingual-e5-large'
suffix   = ST_MODEL[9:]
X_txt_train = np.load(os.path.join(CACHE_DIR, f'X_st_{suffix}_train.npy'))
X_txt_val   = np.load(os.path.join(CACHE_DIR, f'X_st_{suffix}_val.npy'))
X_txt_test  = np.load(os.path.join(CACHE_DIR, f'X_st_{suffix}_test.npy'))
print(f'ST embeddings : train={X_txt_train.shape} | val={X_txt_val.shape}')

ST embeddings : train=(59185, 1024) | val=(12993, 1024)


In [8]:
# ── TF-IDF sparse ────────────────────────────────────────────────────
X_tfidf_train = sp.load_npz(os.path.join(PROCESSED_DIR, 'X_tfidf_train.npz'))
X_tfidf_val   = sp.load_npz(os.path.join(PROCESSED_DIR, 'X_tfidf_val.npz'))
X_tfidf_test  = sp.load_npz(os.path.join(PROCESSED_DIR, 'X_tfidf_test.npz'))
print(f'TF-IDF : train={X_tfidf_train.shape}')

TF-IDF : train=(59185, 50000)


In [9]:
# ── c-TF-IDF (similarites cosinus produit/classe) ────────────────────
X_ctfidf_train = np.load(os.path.join(CACHE_DIR, 'X_ctfidf_train.npy'))
X_ctfidf_val   = np.load(os.path.join(CACHE_DIR, 'X_ctfidf_val.npy'))
X_ctfidf_test  = np.load(os.path.join(CACHE_DIR, 'X_ctfidf_test.npy'))
print(f'c-TF-IDF : {X_ctfidf_train.shape}')

# ── Features texte completes : TF-IDF + c-TF-IDF + e5 ───────────────
X_txt_train_full = sp.hstack([
    X_tfidf_train,
    sp.csr_matrix(X_ctfidf_train),
    sp.csr_matrix(X_txt_train.astype('float32'))])
X_txt_val_full   = sp.hstack([
    X_tfidf_val,
    sp.csr_matrix(X_ctfidf_val),
    sp.csr_matrix(X_txt_val.astype('float32'))])
X_txt_test_full  = sp.hstack([
    X_tfidf_test,
    sp.csr_matrix(X_ctfidf_test),
    sp.csr_matrix(X_txt_test.astype('float32'))])
print(f'Texte complet : {X_txt_train_full.shape}')

c-TF-IDF : (59185, 27)
Texte complet : (59185, 51051)


In [10]:
# ── Embeddings image DINOv2 (backbone gele, pre-calcules) ────────────
IMG_BACKBONE = 'dinov2'
X_img_train = np.load(os.path.join(CACHE_DIR, f'X_img_{IMG_BACKBONE}_train.npy'))
X_img_val   = np.load(os.path.join(CACHE_DIR, f'X_img_{IMG_BACKBONE}_val.npy'))
X_img_test  = np.load(os.path.join(CACHE_DIR, f'X_img_{IMG_BACKBONE}_test.npy'))
X_img_train_n = normalize(X_img_train)
X_img_val_n   = normalize(X_img_val)
X_img_test_n  = normalize(X_img_test)
print(f'Image (DINOv2) : train={X_img_train_n.shape} | val={X_img_val_n.shape}')

Image (DINOv2) : train=(59185, 768) | val=(12993, 768)


## 2. Config OOF

In [11]:
N_CLASSES = 27
N_FOLDS   = 5
OOF_NAME  = f'oof_{N_FOLDS}fold_dino_e5_ctfidf'

# Chemins cache OOF
OOF_TEXT_TRAIN = os.path.join(OOF_DIR, f'{OOF_NAME}_text_train.npy')
OOF_IMG_TRAIN  = os.path.join(OOF_DIR, f'{OOF_NAME}_img_train.npy')

# Probas sur val/test (modeles retrain sur tout le train)
OOF_TEXT_VAL   = os.path.join(OOF_DIR, f'{OOF_NAME}_text_val.npy')
OOF_TEXT_TEST  = os.path.join(OOF_DIR, f'{OOF_NAME}_text_test.npy')
OOF_IMG_VAL    = os.path.join(OOF_DIR, f'{OOF_NAME}_img_val.npy')
OOF_IMG_TEST   = os.path.join(OOF_DIR, f'{OOF_NAME}_img_test.npy')

print(f'Config : {N_FOLDS} folds | nom={OOF_NAME}')

Config : 5 folds | nom=oof_5fold_dino_e5_ctfidf


## 3. OOF — modele texte (LogReg sur TF-IDF + e5)

In [12]:
if os.path.exists(OOF_TEXT_TRAIN):
    P_text_oof = np.load(OOF_TEXT_TRAIN)
    print(f'OOF texte charge : {P_text_oof.shape}')
else:
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    P_text_oof = np.zeros((len(y_train), N_CLASSES), dtype='float32')

    for fold, (idx_tr, idx_va) in enumerate(skf.split(np.zeros(len(y_train)), y_train)):
        print(f'  Fold {fold+1}/{N_FOLDS} — entrainement texte...')
        X_tr_f = X_txt_train_full[idx_tr]
        X_va_f = X_txt_train_full[idx_va]
        y_tr_f = y_train[idx_tr]
        y_va_f = y_train[idx_va]

        clf_f = LogisticRegression(
            C=1.0, max_iter=300, class_weight='balanced',
            solver='saga', n_jobs=-1, random_state=42)
        clf_f.fit(X_tr_f, y_tr_f)

        P_text_oof[idx_va] = clf_f.predict_proba(X_va_f).astype('float32')
        fold_f1 = f1_score(y_va_f, P_text_oof[idx_va].argmax(1), average='macro')
        print(f'    fold {fold+1} val F1 = {fold_f1:.4f}')

    np.save(OOF_TEXT_TRAIN, P_text_oof)
    print(f'OOF texte calcule et sauvegarde : {P_text_oof.shape}')

oof_f1_text = f1_score(y_train, P_text_oof.argmax(1), average='macro')
print(f'OOF F1 texte (train) = {oof_f1_text:.4f}  (estimation non biaisee)')

  Fold 1/5 — entrainement texte...
    fold 1 val F1 = 0.8269
  Fold 2/5 — entrainement texte...
    fold 2 val F1 = 0.8264
  Fold 3/5 — entrainement texte...
    fold 3 val F1 = 0.8377
  Fold 4/5 — entrainement texte...
    fold 4 val F1 = 0.8340
  Fold 5/5 — entrainement texte...
    fold 5 val F1 = 0.8414
OOF texte calcule et sauvegarde : (59185, 27)
OOF F1 texte (train) = 0.8333  (estimation non biaisee)


c:\Users\veyss\anaconda3\envs\rakuten_env\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [13]:
# ── Modele texte final : entraine sur TOUT le train ──────────────────
CLF_TEXT_FULL = os.path.join(MODELS_DIR, f'{OOF_NAME}_clf_text_full.pkl')

if os.path.exists(OOF_TEXT_VAL) and os.path.exists(OOF_TEXT_TEST):
    P_text_val  = np.load(OOF_TEXT_VAL)
    P_text_test = np.load(OOF_TEXT_TEST)
    print('Probas texte val/test chargees depuis cache')
else:
    print('Entrainement modele texte final (full train)...')
    if os.path.exists(CLF_TEXT_FULL):
        with open(CLF_TEXT_FULL, 'rb') as f: clf_text_full = pickle.load(f)
        print('  checkpoint charge')
    else:
        clf_text_full = LogisticRegression(
            C=1.0, max_iter=300, class_weight='balanced',
            solver='saga', n_jobs=-1, random_state=42)
        clf_text_full.fit(X_txt_train_full, y_train)
        with open(CLF_TEXT_FULL, 'wb') as f: pickle.dump(clf_text_full, f)

    P_text_val  = clf_text_full.predict_proba(X_txt_val_full).astype('float32')
    P_text_test = clf_text_full.predict_proba(X_txt_test_full).astype('float32')
    np.save(OOF_TEXT_VAL,  P_text_val)
    np.save(OOF_TEXT_TEST, P_text_test)

val_f1_text = f1_score(y_val, P_text_val.argmax(1), average='macro')
print(f'Texte val F1  = {val_f1_text:.4f}')
print(f'Texte test F1 = {f1_score(y_test, P_text_test.argmax(1), average="macro"):.4f}')

Entrainement modele texte final (full train)...


c:\Users\veyss\anaconda3\envs\rakuten_env\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Texte val F1  = 0.8340
Texte test F1 = 0.8344


## 4. OOF — modele image (tete lineaire sur DINOv2 gele)

In [14]:
IMG_DIM = 768

class _LinearHead(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.fc = nn.Linear(in_dim, N_CLASSES)
    def forward(self, x):
        return self.fc(x)


def train_linear_head(X_tr_np, y_tr_np, X_va_np, y_va_np,
                      ckpt_path, name, n_epochs=40, lr=1e-3, batch_size=256):
    in_dim = X_tr_np.shape[1]
    model  = _LinearHead(in_dim).to(device)

    if os.path.exists(ckpt_path):
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        with torch.no_grad():
            preds = model(torch.tensor(X_va_np, dtype=torch.float32).to(device)).argmax(1).cpu().numpy()
        print(f'{name}: ckpt charge | val_F1={f1_score(y_va_np, preds, average="macro"):.4f}')
        return model

    X_tr = torch.tensor(X_tr_np, dtype=torch.float32)
    y_tr = torch.tensor(y_tr_np, dtype=torch.long)
    X_va = torch.tensor(X_va_np, dtype=torch.float32).to(device)

    loader    = DataLoader(torch.utils.data.TensorDataset(X_tr, y_tr),
                           batch_size=batch_size, shuffle=True)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

    best_f1, no_improve, patience = 0.0, 0, 5
    for epoch in range(n_epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            nn.CrossEntropyLoss()(model(xb), yb).backward()
            optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            preds = model(X_va).argmax(1).cpu().numpy()
        val_f1 = f1_score(y_va_np, preds, average='macro')

        if val_f1 > best_f1:
            best_f1, no_improve = val_f1, 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            no_improve += 1
            if no_improve >= patience:
                break

    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    print(f'{name}: best val_F1 = {best_f1:.4f}')
    return model


def get_probas(model, X_np, batch_size=512):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(X_np), batch_size):
            xb = torch.tensor(X_np[i:i+batch_size], dtype=torch.float32).to(device)
            out.append(torch.softmax(model(xb), dim=1).cpu().numpy())
    return np.vstack(out)

In [15]:
if os.path.exists(OOF_IMG_TRAIN):
    P_img_oof = np.load(OOF_IMG_TRAIN)
    print(f'OOF image charge : {P_img_oof.shape}')
else:
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    P_img_oof = np.zeros((len(y_train), N_CLASSES), dtype='float32')

    for fold, (idx_tr, idx_va) in enumerate(skf.split(np.zeros(len(y_train)), y_train)):
        print(f'  Fold {fold+1}/{N_FOLDS} — entrainement image lineaire...')
        X_tr_f = X_img_train_n[idx_tr]
        X_va_f = X_img_train_n[idx_va]
        y_tr_f = y_train[idx_tr]
        y_va_f = y_train[idx_va]

        ckpt_f = os.path.join(OOF_DIR, f'{OOF_NAME}_img_fold{fold+1}.pth')
        model_f = train_linear_head(
            X_tr_f, y_tr_f, X_va_f, y_va_f,
            ckpt_f, f'img_fold{fold+1}', n_epochs=40, lr=1e-3)

        P_img_oof[idx_va] = get_probas(model_f, X_va_f)
        fold_f1 = f1_score(y_va_f, P_img_oof[idx_va].argmax(1), average='macro')
        print(f'    fold {fold+1} OOF F1 = {fold_f1:.4f}')

    np.save(OOF_IMG_TRAIN, P_img_oof)
    print(f'OOF image calcule et sauvegarde : {P_img_oof.shape}')

oof_f1_img = f1_score(y_train, P_img_oof.argmax(1), average='macro')
print(f'OOF F1 image (train) = {oof_f1_img:.4f}')

  Fold 1/5 — entrainement image lineaire...
img_fold1: best val_F1 = 0.6026
    fold 1 OOF F1 = 0.6026
  Fold 2/5 — entrainement image lineaire...
img_fold2: best val_F1 = 0.6016
    fold 2 OOF F1 = 0.6016
  Fold 3/5 — entrainement image lineaire...
img_fold3: best val_F1 = 0.5898
    fold 3 OOF F1 = 0.5898
  Fold 4/5 — entrainement image lineaire...
img_fold4: best val_F1 = 0.5943
    fold 4 OOF F1 = 0.5943
  Fold 5/5 — entrainement image lineaire...
img_fold5: best val_F1 = 0.5969
    fold 5 OOF F1 = 0.5969
OOF image calcule et sauvegarde : (59185, 27)
OOF F1 image (train) = 0.5972


In [16]:
# ── Modele image final : entraine sur TOUT le train ──────────────────
CKPT_IMG_FULL = os.path.join(OOF_DIR, f'{OOF_NAME}_img_full.pth')

if os.path.exists(OOF_IMG_VAL) and os.path.exists(OOF_IMG_TEST):
    P_img_val  = np.load(OOF_IMG_VAL)
    P_img_test = np.load(OOF_IMG_TEST)
    print('Probas image val/test chargees depuis cache')
else:
    # Utilise la moitie val pour early stopping du modele final
    clf_img_full = train_linear_head(
        X_img_train_n, y_train, X_img_val_n, y_val,
        CKPT_IMG_FULL, f'{OOF_NAME}_img_full', n_epochs=40, lr=1e-3)

    P_img_val  = get_probas(clf_img_full, X_img_val_n)
    P_img_test = get_probas(clf_img_full, X_img_test_n)
    np.save(OOF_IMG_VAL,  P_img_val)
    np.save(OOF_IMG_TEST, P_img_test)

print(f'Image val F1  = {f1_score(y_val, P_img_val.argmax(1), average="macro"):.4f}')
print(f'Image test F1 = {f1_score(y_test, P_img_test.argmax(1), average="macro"):.4f}')

oof_5fold_dino_e5_ctfidf_img_full: best val_F1 = 0.6119
Image val F1  = 0.6119
Image test F1 = 0.6084


## 5. Meta-classifieur entraine sur probas OOF (pas de leakage)

In [17]:
# ── Features meta pour train (OOF = sans leakage) ────────────────────
EXTRA_COLS = [c for c in ['has_desc'] if c in train_split.columns]

if EXTRA_COLS:
    scaler = StandardScaler()
    X_extra_train = scaler.fit_transform(train_split[EXTRA_COLS].fillna(0))
    X_extra_val   = scaler.transform(val_split[EXTRA_COLS].fillna(0))
    X_extra_test  = scaler.transform(test_split[EXTRA_COLS].fillna(0))
else:
    X_extra_train = np.empty((len(train_split), 0))
    X_extra_val   = np.empty((len(val_split),   0))
    X_extra_test  = np.empty((len(test_split),  0))

if 'lang' in train_split.columns:
    ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    X_lang_train = ohe.fit_transform(train_split[['lang']].fillna('fr'))
    X_lang_val   = ohe.transform(val_split[['lang']].fillna('fr'))
    X_lang_test  = ohe.transform(test_split[['lang']].fillna('fr'))
else:
    X_lang_train = np.empty((len(train_split), 0))
    X_lang_val   = np.empty((len(val_split),   0))
    X_lang_test  = np.empty((len(test_split),  0))

# Train : utilise OOF (pas de leakage) — Val/Test : modeles full train
X_meta_train = np.hstack([P_text_oof,  P_img_oof,  X_ctfidf_train, X_extra_train, X_lang_train])
X_meta_val   = np.hstack([P_text_val,  P_img_val,  X_ctfidf_val,   X_extra_val,   X_lang_val])
X_meta_test  = np.hstack([P_text_test, P_img_test, X_ctfidf_test,  X_extra_test,  X_lang_test])

print(f'Meta features : {X_meta_train.shape}')
print(f'  P_text_oof(27) + P_img_oof(27) + c-TF-IDF(27) + extra({X_extra_train.shape[1]}) + lang({X_lang_train.shape[1]})')

Meta features : (59185, 82)
  P_text_oof(27) + P_img_oof(27) + c-TF-IDF(27) + extra(1) + lang(0)


In [18]:
# ── Meta LogReg ───────────────────────────────────────────────────────
CLF_META_LR  = os.path.join(OOF_DIR, f'{OOF_NAME}_meta_logreg.pkl')

if os.path.exists(CLF_META_LR):
    with open(CLF_META_LR, 'rb') as f: meta_lr = pickle.load(f)
    print('meta_lr charge depuis cache')
else:
    print('Entrainement meta LogReg...')
    meta_lr = LogisticRegression(
        C=1.0, max_iter=500, solver='saga', n_jobs=-1, random_state=42)
    meta_lr.fit(X_meta_train, y_train)
    with open(CLF_META_LR, 'wb') as f: pickle.dump(meta_lr, f)

f1_meta_lr_val  = f1_score(y_val,  meta_lr.predict(X_meta_val),  average='macro')
f1_meta_lr_test = f1_score(y_test, meta_lr.predict(X_meta_test), average='macro')
print(f'Meta LogReg  val F1 = {f1_meta_lr_val:.4f} | test F1 = {f1_meta_lr_test:.4f}')

Entrainement meta LogReg...
Meta LogReg  val F1 = 0.8614 | test F1 = 0.8559


In [19]:
# ── Meta MLP ─────────────────────────────────────────────────────────
CLF_META_MLP = os.path.join(OOF_DIR, f'{OOF_NAME}_meta_mlp.pkl')

if os.path.exists(CLF_META_MLP):
    with open(CLF_META_MLP, 'rb') as f: meta_mlp = pickle.load(f)
    print('meta_mlp charge depuis cache')
else:
    print('Entrainement meta MLP...')
    meta_mlp = MLPClassifier(
        hidden_layer_sizes=(128, 64), activation='relu',
        max_iter=300, early_stopping=True,
        validation_fraction=0.1, random_state=42)
    meta_mlp.fit(X_meta_train, y_train)
    with open(CLF_META_MLP, 'wb') as f: pickle.dump(meta_mlp, f)

f1_meta_mlp_val  = f1_score(y_val,  meta_mlp.predict(X_meta_val),  average='macro')
f1_meta_mlp_test = f1_score(y_test, meta_mlp.predict(X_meta_test), average='macro')
print(f'Meta MLP     val F1 = {f1_meta_mlp_val:.4f} | test F1 = {f1_meta_mlp_test:.4f}')

Entrainement meta MLP...
Meta MLP     val F1 = 0.8466 | test F1 = 0.8440


## 6. Moyenne ponderee (baseline rapide)

In [20]:
best_alpha, best_f1_avg = 0.5, 0.0
for alpha in np.arange(0.0, 1.05, 0.05):
    P_avg = alpha * P_text_val + (1 - alpha) * P_img_val
    f1 = f1_score(y_val, P_avg.argmax(1), average='macro')
    if f1 > best_f1_avg:
        best_f1_avg, best_alpha = f1, alpha

P_avg_test = best_alpha * P_text_test + (1 - best_alpha) * P_img_test
print(f'Meilleur alpha (texte) = {best_alpha:.2f}  val F1 = {best_f1_avg:.4f}')
print(f'  (alpha=1.0 = texte seul | alpha=0.0 = image seule)')

Meilleur alpha (texte) = 0.70  val F1 = 0.8492
  (alpha=1.0 = texte seul | alpha=0.0 = image seule)


## 7. Comparaison finale

In [21]:
def _f1(yt, yp):  return round(f1_score(yt, yp, average='macro'),   4)
def _acc(yt, yp): return round(accuracy_score(yt, yp),               4)
def _fw(yt, yp):  return round(f1_score(yt, yp, average='weighted'), 4)

strategies = [
    ('text_only',           P_text_val.argmax(1),                P_text_test.argmax(1)),
    ('image_only',          P_img_val.argmax(1),                 P_img_test.argmax(1)),
    (f'avg_a{best_alpha:.2f}', (best_alpha*P_text_val+(1-best_alpha)*P_img_val).argmax(1),
                             P_avg_test.argmax(1)),
    ('meta_logreg_oof',     meta_lr.predict(X_meta_val),         meta_lr.predict(X_meta_test)),
    ('meta_mlp_oof',        meta_mlp.predict(X_meta_val),        meta_mlp.predict(X_meta_test)),
]

rows = []
for name, pv, pt in strategies:
    rows.append({
        'Strategie': name,
        'F1 val':    _f1(y_val,  pv),
        'F1 test':   _f1(y_test, pt),
        'Acc test':  _acc(y_test, pt),
        'F1w test':  _fw(y_test,  pt),
    })

df_res = pd.DataFrame(rows).sort_values('F1 test', ascending=False)
print('='*62)
print('RESULTATS OOF STACKING')
print('='*62)
print(df_res[['Strategie','F1 val','F1 test','Acc test','F1w test']].to_string(index=False))
print('='*62)
best = df_res.iloc[0]
print(f"Meilleure : {best['Strategie']}  F1={best['F1 test']:.4f}")

RESULTATS OOF STACKING
      Strategie  F1 val  F1 test  Acc test  F1w test
meta_logreg_oof  0.8614   0.8559    0.8618    0.8611
      avg_a0.70  0.8492   0.8467    0.8539    0.8517
   meta_mlp_oof  0.8466   0.8440    0.8508    0.8495
      text_only  0.8340   0.8344    0.8431    0.8418
     image_only  0.6119   0.6084    0.6519    0.6438
Meilleure : meta_logreg_oof  F1=0.8559


In [22]:
# ── Comparaison OOF F1 estime vs val F1 reel ─────────────────────────
print('Estimation OOF (non biaisee) vs Val reel :')
print(f'  Texte  : OOF F1 = {oof_f1_text:.4f}  |  val F1 = {_f1(y_val, P_text_val.argmax(1)):.4f}')
print(f'  Image  : OOF F1 = {oof_f1_img:.4f}  |  val F1 = {_f1(y_val, P_img_val.argmax(1)):.4f}')

Estimation OOF (non biaisee) vs Val reel :
  Texte  : OOF F1 = 0.8333  |  val F1 = 0.8340
  Image  : OOF F1 = 0.5972  |  val F1 = 0.6119


In [23]:
# ── Sauvegarde benchmark ──────────────────────────────────────────────
df_res['Experience'] = OOF_NAME
RESULTS_CSV = os.path.join(MODELS_DIR, f'experiments_{OOF_NAME}_results.csv')
df_res.to_csv(RESULTS_CSV, index=False)
print(f'Resultats sauvegardes : {RESULTS_CSV}')

Resultats sauvegardes : C:/Users/veyss/Nouveau dossier/nov25_alt1_mle_rakuten\models\experiments_oof_5fold_dino_e5_ctfidf_results.csv
